# 用語集を見てみる

`data/glossary.json` を読み込み、登録されている用語を一覧で確認するノートブックです。用語集のファイルは読み取り専用で扱い、内容は変更しません。

上から順にセルを実行してください。最後のセルでは、必要なら特定の回だけに絞り込めます。

## JSONをどう読む？

このファイルは用語の配列だけでなく、`version` や `note` も持つJSONです。そのため、Python標準ライブラリの `json.load(ファイル)` でJSON全体を辞書・リストに読み込み、`terms` の配列を取り出す方法がシンプルです。

- **`json`**：JSONをPythonの辞書・リストに変換する標準ライブラリです。このノートではJSONファイルを読むために使います。
- **`csv`**：CSV形式（行と列の表）を読む標準ライブラリです。今回の入力はJSONなので、`csv` ではなく `json` を使います。
- **pandas**：表データの絞り込み・集計・分析に便利な外部ライブラリです。`DataFrame(terms)` にすれば表形式で見られますが、このリポジトリの必須依存ではありません。用語をまず読むだけなら、pandasを追加せず標準の `json` で十分です。

JSONファイルを読むことと、読み込んだデータを表として分析することは別の段階です。必要になったらpandasなどを使えます。任意の環境でpandasが使える場合は、例えば次のように表示できます。

```python
import pandas as pd
display(pd.DataFrame(terms))
```

## 方法の使い分けと最近の選択肢

| 方法 | 向いている用途 | 今回の用語集では |
|---|---|---|
| 標準 `json` | JSONの階層やメタデータを保って読みたい。追加ライブラリを避けたい | **まずこれを使う**。`version` と `note` を読んだあと `terms` を取り出せる |
| pandas | 表にした後、列選択・並べ替え・絞り込み・集計を続けたい | 任意の選択肢。`terms` を `DataFrame` にする。リポジトリには依存として入っていない |
| 標準 `csv` | 行・列に揃ったCSVファイルを読む、表計算ソフトなどと受け渡す | 入力はJSONなので使わない。JSONをCSVへ変換する必要もない |
| Polars | DataFrame操作が必要で、処理性能や列・スキーマの指定も検討したい | 別の選択肢。今回の65件を眺めるだけなら追加する理由は薄い |

pandasの `read_json()` は、JSONの形がDataFrameに合う場合に便利です。今回のようにメタデータと `terms` 配列が同居するJSONでは、標準 `json.load()` で全体を読み、`terms` を明示して `DataFrame(terms)` にする方が構造が分かりやすくなります。CSVはJSONの別の読み方ではなく、別形式のファイルです。

**最近の動向について確認できた範囲：** pandasとPolarsの公式資料には、JSON/CSVの読み込みやDataFrame操作のAPIがあります。一方、公式API資料だけでは「最近どのライブラリが最も普及しているか」は判断できません。ここでは流行を断定せず、データの形と作業内容で選びます。調査日：2026-11-13。参照した公式資料の表示版は Python 3.15、pandas 3.1.1、Polars 2.1.0 です。これらの版がこのリポジトリの実行環境に入っているという意味ではありません。

- [Python `json`](https://docs.python.org/3/library/json.html)・[Python `csv`](https://docs.python.org/3/library/csv.html)・[Python `collections`](https://docs.python.org/3/library/collections.html)
- [pandas `DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)・[pandas `read_json`](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html)・[pandas `read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html)
- [Polars `read_json`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_json.html)・[Polars `read_csv`](https://docs.pola.rs/api/python/stable/reference/api/polars.read_csv.html)

In [1]:
import json
from pathlib import Path

# 実行場所から親フォルダーをたどり、用語集のあるリポジトリを探します。
search_paths = (Path.cwd(), *Path.cwd().parents)
glossary_path = next(
    (path / "data" / "glossary.json" for path in search_paths
     if (path / "data" / "glossary.json").is_file()),
    None,
)

# ファイルが見つからない場合は、場所の問題を分かる形で知らせます。
if glossary_path is None:
    raise FileNotFoundError(
        "data/glossary.json が見つかりません。リポジトリ内でノートブックを開いてください。"
    )

# UTF-8 の JSON ファイルを開き、Python の辞書・リストとして読み込みます。
with glossary_path.open(encoding="utf-8") as glossary_file:
    glossary_data = json.load(glossary_file)
terms = glossary_data["terms"]

print(f"読み込んだファイル: {glossary_path}")
print(f"バージョン: {glossary_data.get('version', '記載なし')}")
print(f"メモ: {glossary_data.get('note', '記載なし')}")
print(f"用語数: {len(terms)}")
print(f"各用語の項目: {', '.join(terms[0].keys()) if terms else '用語がありません'}")

読み込んだファイル: /workspaces/lecture-ai-engineering-2026/data/glossary.json
バージョン: 2026-draft-1
メモ: AIE2026 第1〜4回・第6回の用語集（たたき台）。講義スライド確定後に内容を差し替える。
用語数: 65
各用語の項目: id, term, aliases, lecture, category, definition, related


### 読み込みセルに出てくるもの

- **`Path`**：ファイルやフォルダーの場所を扱うクラスです。`Path.cwd()` は現在の作業フォルダーを返し、`.parents` はその親フォルダーを順にたどれる値を返します。
- **`next(反復できる値, 見つからない場合の値)`**：先頭の値を1つ取り出します。ここでは、候補の中から `data/glossary.json` が実在する最初のパスを選び、見つからなければ `None` を返します。候補の括弧内は、条件に合うパスを順に作る「ジェネレーター式」です。
- **`Path.is_file()`**：そのパスが実在するファイルかを `True` / `False` で調べます。
- **`Path.open(encoding="utf-8")`**：文字コードをUTF-8に指定してファイルを開きます。`with` を使うと、読み込み後にファイルが自動で閉じられます。
- **`json.load(ファイル)`**：開いたファイルからJSONを読み、Pythonの辞書やリストに変換します。文字列を渡す `json.loads(文字列)` とは引数が異なります。
- **`dict.get(キー, 既定値)`**：キーがあればその値、なければ既定値を返します。バージョンやメモが省略された場合にも表示できるように使っています。
- **`len(値)`**：リストの件数を返します。`terms` の用語数を数えています。

In [2]:
from collections import Counter

# 回ごと・カテゴリごとに件数を数えると、用語集の構成をつかめます。
lecture_counts = Counter(item["lecture"] for item in terms)
category_counts = Counter(item.get("category", "未分類") for item in terms)

print("回ごとの用語数")
for lecture, count in sorted(lecture_counts.items()):
    print(f"第{lecture}回: {count}件")

print("\nカテゴリごとの用語数")
for category, count in sorted(category_counts.items()):
    print(f"{category}: {count}件")

回ごとの用語数
第1回: 26件
第2回: 12件
第3回: 13件
第4回: 7件
第6回: 7件

カテゴリごとの用語数
AIコーディング: 3件
Git/GitHub: 3件
LLM基礎: 7件
MCP: 7件
エージェント: 13件
ファインチューニング: 12件
全体像: 2件
評価: 7件
講座: 4件
開発基礎: 4件
開発環境: 3件


### 件数集計に出てくるもの

- **`collections.Counter(値の集まり)`**：Python標準ライブラリのクラスで、値が何回現れたかを数えます。ここでは各用語の `lecture` や `category` の件数を数えています。
- `item["lecture"] for item in terms` は、用語の一覧から回数だけを1件ずつ取り出すジェネレーター式です。例えば値が `1, 1, 2` なら、`Counter` は「1が2件、2が1件」と数えます。
- **`Counter.items()`**：`(値, 件数)` の組を順に取り出します。**`sorted(...)`** はその組を並べ替え、表示順を安定させます。
- **`item.get("category", "未分類")`**：カテゴリがない用語には「未分類」という値を使います。

In [3]:
from html import escape
from IPython.display import HTML, display


def show_terms(items):
    """用語を読みやすい表にして表示します。"""
    rows = []
    for item in items:
        aliases = "、".join(item.get("aliases", [])) or "なし"
        related = "、".join(item.get("related", [])) or "なし"
        rows.append(
            "<tr>"
            f"<td>{escape(str(item.get('lecture', '')))}</td>"
            f"<td>{escape(str(item.get('category', '')))}</td>"
            f"<td><strong>{escape(str(item.get('term', '')))}</strong><br>"
            f"<small>ID: {escape(str(item.get('id', '')))}</small></td>"
            f"<td>{escape(aliases)}</td>"
            f"<td>{escape(str(item.get('definition', '')))}</td>"
            f"<td>{escape(related)}</td>"
            "</tr>"
        )

    headers = "<tr><th>回</th><th>カテゴリ</th><th>用語</th><th>別名</th><th>定義</th><th>関連ID</th></tr>"
    table = (
        '<table style="border-collapse: collapse; width: 100%;">'
        f"<thead>{headers}</thead><tbody>{''.join(rows)}</tbody></table>"
    )
    display(HTML(table))


# 最初は全件を表示して、用語や定義を眺めます。
show_terms(terms)

回,カテゴリ,用語,別名,定義,関連ID
1,全体像,AIエンジニアリングID: ai-engineering,AI Engineering,学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。,llm、evaluation-loop
1,全体像,評価と改善のサイクルID: evaluation-loop,改善サイクル,作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。,ai-engineering、hallucination
1,LLM基礎,LLM（大規模言語モデル）ID: llm,大規模言語モデル、Large Language Model,大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。,token、prompt、pretraining
1,LLM基礎,トークンID: token,token,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,context-window、rate-limit
1,LLM基礎,コンテキストウィンドウID: context-window,コンテキスト長、context window,LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。,token、agent-loop
1,LLM基礎,プロンプトID: prompt,prompt,LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。,system-prompt、llm
1,LLM基礎,システムプロンプトID: system-prompt,指示文、instructions、system instruction,利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。,prompt、guardrail
1,LLM基礎,temperatureID: temperature,温度パラメータ,次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。,llm、evaluation-loop
1,LLM基礎,ハルシネーションID: hallucination,幻覚、hallucination,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,tool、evaluation-loop
1,開発基礎,APIID: api,Application Programming Interface、Web API,プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。,api-key、rate-limit


### 表示用コードに出てくるもの

- **`html.escape(文字列)`**：文字列中の `<` や `&` などをHTML用の表記に置き換えます。既定の `quote=True` では引用符も変換します。用語の定義にHTMLタグのような文字が含まれてもタグとして扱われず、文章として読めます。例えば `<` は `&lt;` になります。
- **`str(値)`**：数値などを文字列に変換します。`escape` に渡す値を文字列にそろえるために使います。
- **`list.append(値)`**：リストの末尾に値を追加します。ここでは用語ごとに作った表の行を `rows` にためています。
- **`"、".join(文字列のリスト)`**：リストの文字列を「、」で連結します。別名や関連IDを1つのセルに表示するために使います。
- **`IPython.display.HTML(HTML文字列)`**：JupyterにHTMLとして表示する内容を渡すクラスです。データ由来の文字列は先に `escape` し、表のマークアップと混ざらないようにしています。
- **`IPython.display.display(値)`**：値をノートブック上に表示します。ここでは `HTML` オブジェクトを表として描画させています。

このセルでHTML表を自分で組み立てているのは表示を整えるためで、JSONを読むために必要な処理ではありません。pandasを使える環境なら、先ほどの `display(pd.DataFrame(terms))` の方が表の表示を短く書けます。

In [8]:
# 特定の回だけ見たい場合は、None を回数に置き換えます（例: 2）。
lecture_filter = 4

# None なら全件、数字ならその回の用語だけを表示します。
filtered_terms = (
    terms
    if lecture_filter is None
    else [item for item in terms if item["lecture"] == lecture_filter]
)
print(f"表示件数: {len(filtered_terms)}")
show_terms(filtered_terms)

表示件数: 7


回,カテゴリ,用語,別名,定義,関連ID
4,評価,評価セットID: evaluation-dataset,評価データセット、eval set、テストセット,評価のために用意した「入力と、期待する出力または合格の条件」の集まり。正常系だけでなく、壊れやすい入力（エッジケース）を意図的に含めて、どこで失敗するかを見る。本講座の第4回演習では、講師が用意した10件に受講生がラベルを付ける。,edge-case、rubric、llm-as-a-judge
4,評価,エッジケースID: edge-case,edge case、境界ケース,入力の境界や想定外の形。空の入力、極端に長い入力、表記ゆれ、範囲外の質問、指示を上書きしようとする要求など。評価セットに意図的に含め、採点器やエージェントがそこでどうふるまうかを確かめる。,evaluation-dataset
4,評価,ルーブリックID: rubric,rubric、評価基準、採点基準,合格・不合格を判定するための基準を、項目ごとに文章で定めたもの。人にもLLMにも同じ基準を渡して採点させる。基準に書いていないことは人もLLMも決められないので、判断が分かれた項目は方針を決めて基準に書き足す。本講座では data/eval/rubric.md がこれにあたる。,evaluation-dataset、llm-as-a-judge、needs-review
4,評価,LLM-as-a-JudgeID: llm-as-a-judge,LLM採点、LLM judge、自動採点,採点の基準（ルーブリック）を与えたLLMに、別のモデルの出力を判定させる方法。大量の出力を安く速く採点できるが、採点役の癖（冗長な回答を好む、順序に影響されるなど）が結果に混ざる。人の判定との一致率で確かめてから使う。,rubric、agreement-rate、position-bias、structured-output
4,評価,一致率ID: agreement-rate,agreement rate、人との一致、inter-rater agreement,2つの判定（人とLLM、人と人）が同じ結果になった件数の割合。採点器を信じてよいかの根拠になる。主観的な基準は人同士でも割れるので、一致率の天井は100%ではない。,llm-as-a-judge、rubric
4,評価,要確認ID: needs-review,unsure、人手確認、要レビュー,自動判定では決められない項目（LLMが unsure と答えた項目）を、人が見る列に回すこと。一致率の計算からは外す。要確認が出た項目は、基準に書いていなかったことの印なので、方針を決めて基準に書き足す。,rubric、llm-as-a-judge
4,評価,位置バイアスID: position-bias,position bias、順序バイアス,2つの回答を並べて「どちらが良いか」を判定させるとき、内容ではなく提示された順序に判定が引きずられる採点器の癖。順序を入れ替えて同じ結果になるかで確かめる。長い回答を好む冗長性バイアスと並ぶ代表的な癖。,llm-as-a-judge
